# Session 01 · Introduction to T-SQL

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - connect to the sample campus database and list its tables

In [2]:
import sqlite3
from campus_db import connect, show

con = connect()
show(con, "SELECT name AS table_name FROM sqlite_master WHERE type = 'table' ORDER BY name")
# T-SQL: SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES;

table_name
----------
batches   
faculty   
marks     
students  
(4 rows)



[('batches',), ('faculty',), ('marks',), ('students',)]

## Example 2 - DDL: create a table with data types and constraints

In [3]:
con.execute("""
CREATE TABLE courses (
    course_code  TEXT    PRIMARY KEY,           -- T-SQL: VARCHAR(10) NOT NULL PRIMARY KEY
    title        TEXT    NOT NULL,              -- T-SQL: NVARCHAR(80) NOT NULL
    duration_wk  INTEGER CHECK (duration_wk > 0),
    fee          INTEGER DEFAULT 0
)""")
show(con, "PRAGMA table_info(courses)")          # T-SQL: EXEC sp_help 'dbo.courses';

cid | name        | type    | notnull | dflt_value | pk
----+-------------+---------+---------+------------+---
0   | course_code | TEXT    | 0       | None       | 1 
1   | title       | TEXT    | 1       | None       | 0 
2   | duration_wk | INTEGER | 0       | None       | 0 
3   | fee         | INTEGER | 0       | 0          | 0 
(4 rows)



[(0, 'course_code', 'TEXT', 0, None, 1),
 (1, 'title', 'TEXT', 1, None, 0),
 (2, 'duration_wk', 'INTEGER', 0, None, 0),
 (3, 'fee', 'INTEGER', 0, '0', 0)]

## Example 3 - DML: insert and read rows

In [4]:
con.executemany("INSERT INTO courses VALUES (?, ?, ?, ?)", [
    ("DAI", "Diploma in AI", 48, 180000),
    ("DS", "Data Science", 24, 90000),
    ("PY", "Python Basics", 8, 25000),
])
show(con, "SELECT * FROM courses")
show(con, "SELECT title, fee FROM courses WHERE fee > 50000")

course_code | title         | duration_wk | fee   
------------+---------------+-------------+-------
DAI         | Diploma in AI | 48          | 180000
DS          | Data Science  | 24          | 90000 
PY          | Python Basics | 8           | 25000 
(3 rows)

title         | fee   
--------------+-------
Diploma in AI | 180000
Data Science  | 90000 
(2 rows)



[('Diploma in AI', 180000), ('Data Science', 90000)]

## Example 4 - constraints protect the data

In [5]:
for bad in [("PY", "Duplicate code", 4, 0), ("XX", None, 4, 0), ("YY", "Negative", -2, 0)]:
    try:
        con.execute("INSERT INTO courses VALUES (?, ?, ?, ?)", bad)
    except sqlite3.IntegrityError as e:
        print(f"{bad[0]}: rejected -> {e}")

PY: rejected -> UNIQUE constraint failed: courses.course_code
XX: rejected -> NOT NULL constraint failed: courses.title
YY: rejected -> CHECK constraint failed: duration_wk > 0


## Example 5 - your first real queries on the campus data

In [6]:
show(con, "SELECT name, city FROM students")
show(con, "SELECT name, city, fee_paid FROM students WHERE city = 'Karachi' ORDER BY fee_paid DESC")
show(con, "SELECT COUNT(*) AS total_students FROM students")

name          | city     
--------------+----------
Ayesha Khan   | Karachi  
Bilal Ahmed   | Lahore   
Hira Siddiqui | Karachi  
Omar Farooq   | Quetta   
Zara Hussain  | Karachi  
Hamza Malik   | Islamabad
Fatima Noor   | Lahore   
Ali Raza      | None     
Mariam Javed  | Karachi  
Saad Qureshi  | Peshawar 
(10 rows)

name          | city    | fee_paid
--------------+---------+---------
Ayesha Khan   | Karachi | 45000   
Hira Siddiqui | Karachi | 45000   
Zara Hussain  | Karachi | 45000   
Mariam Javed  | Karachi | 15000   
(4 rows)

total_students
--------------
10            
(1 rows)



[(10,)]

## Example 6 - DCL / TCL: a transaction you can roll back

In [7]:
con.execute("BEGIN")                                  # T-SQL: BEGIN TRANSACTION;
con.execute("DELETE FROM courses")
print("inside transaction:", con.execute("SELECT COUNT(*) FROM courses").fetchone()[0])
con.execute("ROLLBACK")                               # T-SQL: ROLLBACK TRANSACTION;
print("after rollback   :", con.execute("SELECT COUNT(*) FROM courses").fetchone()[0])

inside transaction: 0
after rollback   : 3
